In [1]:
import pandas as pd
import os
import funciones

Cargamos el archivo `corpus.csv` y visualizamos las primeras filas para ver su estructura.

In [2]:
ruta_input = "corpus.csv"

if os.path.exists(ruta_input):
    df = pd.read_csv(ruta_input, encoding='utf-8')
    print(f"Se cargaron {len(df)} películas.")
    display(df.head(3))
else:
    print(f"Error: No se encontró el archivo '{ruta_input}' en el directorio.")

Se cargaron 70 películas.


,titulo,idioma,sinopsis,reseña,género
0,Avatar,Español,Exploramos en Avatar la historia de una serie ...,"Tras verla, diría que es un poco lenta, pero c...","Acción, Aventura, Fantasía, Ciencia Ficción"
1,Pirates of the Caribbean: At World's End,Inglés,Acompañamos a los personajes de Pirates of the...,"Desde mi punto de vista, estamos ante impactan...","Aventura, Fantasía, Acción"
2,Spectre,Español,Acompañamos a los personajes de Spectre en un ...,Una experiencia cinematográfica fascinante y m...,"Acción, Aventura, Crimen"


Para cada película, unimos los siguientes campos en un único texto:
- `titulo` (Título de la película)
- `idioma` (Idioma de la película)
- `sinopsis` (Sinopsis de la película)
- `reseña` (Reseña de los usuarios)
- `género` (Géneros de la película)

Reemplazamos posibles valores nulos (`NaN`) con cadenas vacías para evitar errores de concatenación.
Además, aplicamos técnicas de normalización usando NLTK (conversión a minúsculas, tokenización, eliminación de signos de puntuación y palabras vacías, y reducción a la raíz o stemming).

In [3]:
df = df.fillna("")

df['documento_crudo'] = (
    df['titulo'] + 
    ". " + df['idioma'] + 
    " " + df['sinopsis'] + 
    " " + df['reseña'] + 
    " " + df['género']
)

df['documento'] = df['documento_crudo'].apply(funciones.procesar_texto)

In [4]:
print(f"Película: {df['titulo'].iloc[0]}\n")
print("Documento generado para búsqueda:")
print(df['documento'].iloc[0])

Película: Avatar

Documento generado para búsqueda:
avat español explor avat histori seri event misteri mantien suspens final ideal amant cin mensaj potent tras verl dir lent actuacion brillant recomend accion aventur fantas cienci ficcion


Guardamos el DataFrame resultante en un nuevo archivo CSV.

In [5]:
ruta_output = "corpus_procesado.csv"
df.to_csv(ruta_output, index=False, encoding='utf-8')
print(f" El corpus procesado se guardó en: {ruta_output}")

 El corpus procesado se guardó en: corpus_procesado.csv


### Creación del Índice Invertido con Whoosh
Utilizamos la biblioteca `Whoosh` para construir un índice invertido. Definiremos un esquema con el título y el contenido procesado, crearemos un directorio para el índice y añadiremos cada documento del corpus.

In [6]:
import os
from whoosh.index import create_in
from whoosh.fields import Schema, TEXT, ID

schema = Schema(
    id=ID(stored=True, unique=True),
    titulo=TEXT(stored=True),
    idioma=TEXT(stored=True),
    sinopsis=TEXT(stored=True),
    reseña=TEXT(stored=True),
    genero=TEXT(stored=True),
    contenido=TEXT(stored=True)
)

index_dir = "indexdir"
if not os.path.exists(index_dir):
    os.mkdir(index_dir)

ix = create_in(index_dir, schema)
writer = ix.writer()

for i, row in df.iterrows():
    writer.add_document(
        id=str(i),
        titulo=str(row['titulo']),
        idioma=str(row['idioma']),
        sinopsis=str(row['sinopsis']),
        reseña=str(row['reseña']),
        genero=str(row['género']),
        contenido=str(row['documento'])
    )

writer.commit()
print("Índice invertido creado con éxito en el directorio 'indexdir'.")

Índice invertido creado con éxito en el directorio 'indexdir'.


### Ejemplos de Búsqueda
A continuación, probamos las funciones de búsqueda booleana y vectorial.

In [ ]:
# Prueba del Modelo Booleano
query_bool = "aventura AND ( espacio OR romance )"
print("Query Booleana:", query_bool)
resultados_bool = funciones.obtener_resultados_booleanos(query_bool, index_dir="indexdir")
print("IDs recuperados (Booleano):", resultados_bool)
if resultados_bool:
    print("\nPelículas recuperadas:")
    display(df.iloc[list(resultados_bool)][['titulo', 'género']])
else:
    print("No se encontraron resultados.")

Query Booleana: aventura AND ( espacio OR romance )
IDs recuperados (Booleano): {50}

Películas recuperadas:


,titulo,género
50,Prince of Persia: The Sands of Time,"Aventura, Fantasía, Acción, Romance"


In [11]:
# Prueba del Modelo Vectorial
documentos = [{'id': i, 'contenido': row['documento']} for i, row in df.iterrows()]

query_vect = "película de ciencia ficción con batallas espaciales"
print("Query Vectorial:", query_vect)
resultados_vect = funciones.obtener_resultados_coseno(query_vect, documentos)
print(f"Se han encontrado {len(resultados_vect)} películas")

if resultados_vect:
    print("\nTop 5 Películas recuperadas:")
    display(df.iloc[resultados_vect[:5]][['titulo', 'género']])
else:
    print("No se encontraron resultados.")

Query Vectorial: película de ciencia ficción con batallas espaciales
Se han encontrado 43 películas

Top 5 Películas recuperadas:


,titulo,género
16,The Avengers,"Ciencia Ficción, Acción, Aventura"
35,Transformers: Revenge of the Fallen,"Ciencia Ficción, Acción, Aventura"
31,Iron Man 3,"Acción, Aventura, Ciencia Ficción"
45,World War Z,"Acción, Drama, Terror, Ciencia Ficción, Suspense"
64,X-Men: Apocalypse,Ciencia Ficción
